# SETUP AND DEPENDENCIES

In [1]:
!pip install stanza

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------- ----------- 0.8/1.1 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/590.6 kB ? eta -:--:--
   ---------------------------------------- 590.6/590.6 kB 4.3 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ------------- -------------------------- 1/3 [emoji]
   -------------------------- ------------- 2/3 [stanza]
   -------------------------- ------------- 2/3 [stanza]
   -------------------------- ------------- 


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb

In [3]:
from tqdm import tqdm
tqdm.pandas() 

# DATA LOADING

In [60]:
train_df = pd.read_csv("D:/. Lomba/objective-quest-2025/train.csv")
test_df = pd.read_csv("D:/. Lomba/objective-quest-2025/test.csv")

# Mark train vs test
train_df["is_train"] = 1
test_df["is_train"] = 0
test_df["lama hukuman (bulan)"] = None

In [53]:
base_path = "D:/. Lomba/objective-quest-2025/file_putusan/file_putusan/"

def load_texts(ids):
    texts = []
    for case_id in ids:
        file_path = os.path.join(base_path, f"{case_id}.txt")
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                texts.append(f.read())
        except:
            texts.append("")
    return texts

# Load texts into each df
train_df["text"] = load_texts(train_df["id"])
test_df["text"] = load_texts(test_df["id"])


In [63]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16572 entries, 0 to 16571
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    16572 non-null  object
 1   lama hukuman (bulan)  16572 non-null  int64 
 2   is_train              16572 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 388.5+ KB


In [64]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7103 entries, 0 to 7102
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    7103 non-null   object
 1   is_train              7103 non-null   int64 
 2   lama hukuman (bulan)  0 non-null      object
dtypes: int64(1), object(2)
memory usage: 166.6+ KB


In [45]:
train = train_df.copy()
test = test_df.copy()
df = pd.concat([train, test], ignore_index=True)

In [46]:
df.tail(5)

,lama hukuman (bulan),indobert_0,indobert_1,indobert_2,indobert_3,indobert_4,indobert_5,indobert_6,indobert_7,indobert_8,...,svd_293,svd_294,svd_295,svd_296,svd_297,svd_298,svd_299,id,is_train,text
10261,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doc_1271,0.0,hkama\nahkamah Agung Repub\nahkamah Agung Repu...
10262,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doc_9130,0.0,hkama\nahkamah Agung Repub\nahkamah Agung Repu...
10263,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doc_4853,0.0,hkama\nahkamah Agung Repub\nahkamah Agung Repu...
10264,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doc_1692,0.0,hkama\nahkamah Agung Repub\nahkamah Agung Repu...
10265,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doc_20539,0.0,hkama\nahkamah Agung Repub\nahkamah Agung Repu...


In [43]:
# Save the combined dataframe with text and flag
train_df.to_csv("train_with_text.csv", index=False)
test_df.to_csv("test_with_text.csv", index=False)

KeyboardInterrupt: 

In [4]:
train_df = pd.read_csv("train_with_text.csv")
test_df = pd.read_csv("test_with_text.csv")

In [5]:
df = pd.read_csv("clean_text.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23238 entries, 0 to 23237
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    23238 non-null  object 
 1   lama hukuman (bulan)  16572 non-null  float64
 2   is_train              23238 non-null  int64  
 3   text                  23238 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 726.3+ KB


# TEXT PROCESSING

In [17]:
import re

def clean_court_text(text: str) -> str:
    text = str(text)

    # --- 1. Remove noisy "Mahkamah Agung" OCR fragments ---
    text = re.sub(
        r"h?k?m?a?h?ma?h?.{0,10}agung.{0,50}indonesi[ai]?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # catch shorter junk like "'hkama a blik indonesi"
    text = re.sub(r"[\'\"]?hk?ma\w*\s+a\s+blik\s+indonesi", " ", text, flags=re.IGNORECASE)

    # --- 2. Remove disclaimers & metadata ---
    text = re.sub(r"Direktori Putusan.*", " ", text)
    text = re.sub(r"DEMI KEADILAN BERDASARKAN KETUHANAN.*", " ", text)
    text = re.sub(r"Disclaimer.*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Halaman\s+\d+.*", " ", text)
    text = re.sub(r"Nomor\s*[:].*?\n", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Nomor Perkara.*", " ", text, flags=re.IGNORECASE)

    # --- 3. Mask personal identifiers ---
    text = re.sub(r"\bNama Lengkap\s*:.*?(?=[0-9]+\.)", " TERDAKWA ", text, flags=re.IGNORECASE)
    text = re.sub(r"Nomor Identitas\s*:\s*\d+", " [ID] ", text, flags=re.IGNORECASE)
    text = re.sub(r"\d{1,2}\s+\w+\s+\d{4}", " [DOB] ", text)       # e.g. "12 Maret 1972"
    text = re.sub(r"\d{1,2}[-/]\d{1,2}[-/]\d{2,4}", " [DOB] ", text) # e.g. "12-03-1972"
    text = re.sub(r"Umur[^,;.]+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Alamat.*?(?=[0-9]+\.)", " [ALAMAT] ", text, flags=re.IGNORECASE)

    # --- 4. Remove email & telp ---
    text = re.sub(r"Email\s*:\s*\S+@\S+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Telp\s*[:.]?\s*\+?\d[\d\-\s\(\)]+", " ", text, flags=re.IGNORECASE)

    # --- 5. Preserve useful numbers ---
    text = re.sub(r"Rp\s?[\d\.\,]+", lambda m: "Rp" + re.sub(r"[^\d]", "", m.group(0)), text)
    text = re.sub(r"(\d+)\s*\((.*?)\)", r"\1", text)  # "9 (sembilan)" → 9

    # --- 6. Clean OCR junk / fix spacing ---
    text = re.sub(r"\b(\w+)(\s+\1)+\b", r"\1", text)  # collapse repeated words
    text = re.sub(r"-\s*\n", "", text)                # fix hyphen line breaks
    text = re.sub(r"\s*\n\s*", " ", text)             # remove bad newlines
    text = re.sub(r"[^\x00-\x7F]+", " ", text)        # remove stray unicode
    text = re.sub(r"\s+", " ", text)                  # normalize whitespace

    # --- 7. Remove very short tokens (≤2 chars, keep digits like "12") ---
    text = " ".join([w for w in text.split() if len(w) > 2 or w.isdigit()])

    return text.strip().lower()

from tqdm import tqdm
tqdm.pandas()

df["text"] = df["text"].progress_apply(clean_court_text)

100%|████████████████████████████████████████████████████████████████████████████| 23238/23238 [08:58<00:00, 43.13it/s]


In [30]:
import re
from tqdm import tqdm

def clean_again(text: str) -> str:
    text = str(text).lower()

    # Remove specific unwanted phrase
    text = re.sub(r"mahkamah indonesia", " ", text)

    # Keep only alphanumeric characters (replace others with space)
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Remove short tokens (<=2 chars unless digit)
    text = " ".join([w for w in text.split() if len(w) > 2 or w.isdigit()])

    return text.strip()

tqdm.pandas()
df["text"] = df["text"].progress_apply(clean_again)

100%|███████████████████████████████████████████████████████████████████████████| 23238/23238 [00:46<00:00, 503.80it/s]


In [7]:
ocr_norm_dict = {
    # --- Mahkamah / Indonesia OCR variants ---
    "hkama": "mahkamah",
    "hkmah": "mahkamah",
    "mhkm": "mahkamah",
    "ahkamah": "mahkamah",
    "mah": "mahkamah",   # when truncated

    "blik": "indonesia",
    "indonsia": "indonesia",
    "indonsi": "indonesia",
    "indons": "indonesia",
    "indonesi": "indonesia",
    "indnesi": "indonesia",

    # --- Common short artifacts ---
    "repoblik": "republik",
    "repob": "republik",
    "repub": "republik",
    "repubi": "republik",

    # --- Case headers repeating ---
    "kepaniteraan": "kepaniteraan",
    "putusan": "putusan",
    "putusanma": "putusan",
    "putusanmahmah": "putusan",
}

In [8]:
import re

def normalize_ocr(text: str, norm_dict: dict) -> str:
    text = text.lower()
    # replace all noisy tokens by clean ones
    for noisy, clean in norm_dict.items():
        text = re.sub(rf"\b{noisy}\b", clean, text)
    return text

df["text"] = df["text"].apply(lambda x: normalize_ocr(x, ocr_norm_dict))

In [31]:
df.iloc[1,3]

'terdakwa lahir yahadian jenis kelamin laki laki kebangsaan indonesia tinggal kampung yahadian 001 kel yahadian kec kais kab sorong selatan agama kristen pekerjaan nelayan perikanan terdakwa lukas regoy ditahan tahanan penyidik oleh penyidik tanggal dob tanggal dob penyidik perpanjangan penuntut tanggal dob tanggal dob penyidik perpanjangan ketua pengadilan negeri tanggal dob tanggal dob penuntut tanggal dob tanggal dob penyidik perpanjangan ketua pengadilan negeri tanggal dob tanggal dob hakim pengadilan negeri tanggal dob tanggal dob hakim pengadilan negeri perpanjangan ketua pengadilan negeri tanggal dob tanggal dob hakim pengadilan negeri perpanjangan ketua pengadilan papua barat tanggal dob tanggal dob terdakwa menghadap dipersidangan damping penasehat hukum fernando marthin ginuni frans daniel wattimena ruth hana ohee advokad law office fernando ginuny partners advocate legal consultans ber alamat 5 000 lima ribu rupiah mendengar pembelaan terdakwa penasihat hukum terdakwa pokokn

In [19]:
!pip install Sastrawi


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
# from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# factory = StemmerFactory()
# stemmer = factory.create_stemmer()

# def stem_indonesian(text: str) -> str:
#     return stemmer.stem(text)

# from tqdm import tqdm
# tqdm.pandas()

# df["text"] = df["text"].progress_apply(stem_indonesian)

  0%|                                                                             | 2/23238 [00:30<97:00:51, 15.03s/it]


KeyboardInterrupt: 

In [24]:
!pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 2.4 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.5 MB 2.0 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [26]:
stop_words = set(stopwords.words("indonesian"))

In [27]:
def remove_stopwords(text: str, lang="indonesian") -> str:
    # Load stopwords
    sw = set(stopwords.words(lang))
    
    # Tokenize by whitespace
    tokens = text.split()
    
    # Keep only non-stopwords
    filtered = [w for w in tokens if w not in sw]
    
    return " ".join(filtered)

tqdm.pandas()

df["text"] = df["text"].progress_apply(remove_stopwords)


100%|███████████████████████████████████████████████████████████████████████████| 23238/23238 [00:33<00:00, 703.83it/s]


In [54]:
# Reselect just the important columns
df = df[["id", "is_train", "lama hukuman (bulan)", "text"]]

df.head()

,id,is_train,lama hukuman (bulan),text
0,doc_13590,1,10.0,terdakwa terdakwa julianti lahinta alias jurni...
1,doc_14914,1,60.0,terdakwa lahir yahadian jenis kelamin laki lak...
2,doc_21900,1,18.0,acara pemeriksaan peradilan tingkat pertama me...
3,doc_14859,1,72.0,terdakwa lahir manado jenis kelamin laki laki ...
4,doc_10962,1,60.0,terdakwa lahir bengkulu jenis kelamin laki lak...


# FEATURE ENGINEERING

## TF-IDF Word Frequency

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd

# Step 1: TF-IDF (sparse, efficient)
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["text"])   # stays sparse

# Step 2: SVD directly on sparse matrix
svd = TruncatedSVD(n_components=10000, random_state=42)  # pick #components << 10000
tfidf_reduced = svd.fit_transform(tfidf_matrix)

# Step 3: Convert to DataFrame (much smaller!)
tfidf_df = pd.DataFrame(
    tfidf_reduced,
    columns=[f"svd_{i}" for i in range(tfidf_reduced.shape[1])],
    index=df.index
)

In [49]:
tfidf_df.head(5)

,svd_0,svd_1,svd_2,svd_3,svd_4,svd_5,svd_6,svd_7,svd_8,svd_9,...,svd_9990,svd_9991,svd_9992,svd_9993,svd_9994,svd_9995,svd_9996,svd_9997,svd_9998,svd_9999
0,0.342306,0.199459,-0.096427,-0.197897,0.113237,0.025441,-0.050520,-0.018167,0.071074,-0.012903,...,0.000374,-0.005500,0.002473,-0.002161,-0.002066,0.002409,0.003030,0.000487,0.000526,0.002858
1,0.357488,0.237120,-0.065348,-0.208019,0.166673,0.008000,-0.010674,-0.001212,0.114597,0.020869,...,-0.000681,-0.001132,0.001282,0.000290,0.001872,0.000899,0.002815,-0.001059,-0.002703,0.002615
2,0.312037,0.042014,-0.069554,-0.068293,-0.093098,-0.019683,-0.000706,0.000646,-0.031740,0.010606,...,0.002515,0.002831,-0.001631,0.000071,0.003285,0.001845,0.000105,0.001532,-0.001475,0.000601
3,0.395324,-0.147497,0.064375,-0.033740,-0.004510,-0.165504,-0.101923,-0.049732,-0.037278,-0.057599,...,-0.001689,0.001129,-0.003002,0.003481,0.002204,-0.004006,0.000914,-0.001327,-0.003738,-0.000344
4,0.588246,-0.313942,0.190750,0.009566,0.016638,-0.071839,-0.052052,-0.049449,0.026939,-0.032332,...,0.003581,-0.001232,-0.001316,-0.003664,-0.003823,0.000064,-0.000351,0.000571,-0.001696,0.001344


In [51]:
tfidf_df.to_csv("tfidf.csv", index=False)

## IndoBERT Vectorization

In [35]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers.utils import logging as hf_logging
import warnings

# Suppress HuggingFace and Python warnings
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# --- Load IndoBERT-large ---
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-large-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-large-p1").to("cuda")

def indobert_chunk_embed(texts, max_length=512, batch_size=16, stride=0.8, use_fp16=True):
    """
    Faster IndoBERT-large embeddings with batching + optional fp16.
    """
    model.eval()
    final_embeddings = []

    dtype = torch.float16 if use_fp16 else torch.float32

    with torch.no_grad():
        for text in tqdm(texts, desc="IndoBERT-large Embeddings", total=len(texts)):
            # Tokenize with overflowing chunks
            tokens = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                stride=int(max_length * (1 - stride)),
                return_overflowing_tokens=True,
                padding="max_length",
                return_tensors="pt"
            )

            input_ids = tokens["input_ids"].to("cuda")
            attention_mask = tokens["attention_mask"].to("cuda")

            # Process in batches
            chunk_embeddings = []
            for i in range(0, input_ids.size(0), batch_size):
                batch_ids = input_ids[i:i+batch_size]
                batch_mask = attention_mask[i:i+batch_size]

                outputs = model(
                    input_ids=batch_ids,
                    attention_mask=batch_mask
                )

                emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
                chunk_embeddings.append(emb.to(dtype).cpu().numpy())

            # Average across chunks for final doc vector
            doc_embedding = np.mean(np.vstack(chunk_embeddings), axis=0)
            final_embeddings.append(doc_embedding)

    return np.vstack(final_embeddings)



# --- Example usage ---
embeddings = indobert_chunk_embed(df["text"].tolist(), max_length=512, stride=0.8, batch_size=16)

# Put into DataFrame (1024-dim for large)
emb_df = pd.DataFrame(
    embeddings,
    columns=[f"indobert_{i}" for i in range(embeddings.shape[1])],
    index=df.index
)

IndoBERT-large Embeddings: 100%|███████████████████████████████████████████████| 23238/23238 [3:58:44<00:00,  1.62it/s]


In [36]:
emb_df.head(5)

,indobert_0,indobert_1,indobert_2,indobert_3,indobert_4,indobert_5,indobert_6,indobert_7,indobert_8,indobert_9,...,indobert_1014,indobert_1015,indobert_1016,indobert_1017,indobert_1018,indobert_1019,indobert_1020,indobert_1021,indobert_1022,indobert_1023
0,0.523438,0.229004,-0.600586,0.078857,0.000725,-0.063354,0.282959,-0.047363,-0.290283,-0.564453,...,-0.310059,-0.302246,-0.429688,0.082214,0.116516,-0.052307,-1.208008,0.242920,-0.913086,1.100586
1,0.615234,0.380615,-0.715332,0.347900,-0.187988,-0.221191,0.479248,-0.294678,-0.374512,-0.201660,...,-0.258545,-0.725586,-0.371582,0.133545,0.408203,-0.374023,-1.507812,0.250977,-0.851562,0.735840
2,0.689453,-0.693848,-0.737305,0.776367,-0.001667,0.519531,0.716797,-0.863281,-0.223022,0.067688,...,-0.310791,0.014183,-0.216553,0.476562,-0.024582,-0.564453,-0.868164,-0.198486,-1.140625,1.036133
3,0.492188,0.087280,-0.687500,0.322021,-0.009773,-0.306396,0.562500,-0.370117,-0.354004,-0.386475,...,-0.295654,-0.734863,-0.310303,0.007412,0.553223,-0.363037,-0.997070,-0.321777,-1.227539,0.844727
4,0.494385,0.137939,-0.618164,0.332520,-0.020767,-0.322998,0.519531,-0.427490,-0.153198,-0.184448,...,-0.390137,-0.697266,-0.335205,0.253174,0.547363,-0.022079,-1.272461,-0.301514,-1.000977,0.931641


In [37]:
emb_df.to_csv('indobert.csv', index=False)

In [55]:
df = pd.concat([df, emb_df, tfidf_df], axis=1)

df.head()

,id,is_train,lama hukuman (bulan),text,indobert_0,indobert_1,indobert_2,indobert_3,indobert_4,indobert_5,...,svd_9990,svd_9991,svd_9992,svd_9993,svd_9994,svd_9995,svd_9996,svd_9997,svd_9998,svd_9999
0,doc_13590,1,10.0,terdakwa terdakwa julianti lahinta alias jurni...,0.523438,0.229004,-0.600586,0.078857,0.000725,-0.063354,...,0.000374,-0.005500,0.002473,-0.002161,-0.002066,0.002409,0.003030,0.000487,0.000526,0.002858
1,doc_14914,1,60.0,terdakwa lahir yahadian jenis kelamin laki lak...,0.615234,0.380615,-0.715332,0.347900,-0.187988,-0.221191,...,-0.000681,-0.001132,0.001282,0.000290,0.001872,0.000899,0.002815,-0.001059,-0.002703,0.002615
2,doc_21900,1,18.0,acara pemeriksaan peradilan tingkat pertama me...,0.689453,-0.693848,-0.737305,0.776367,-0.001667,0.519531,...,0.002515,0.002831,-0.001631,0.000071,0.003285,0.001845,0.000105,0.001532,-0.001475,0.000601
3,doc_14859,1,72.0,terdakwa lahir manado jenis kelamin laki laki ...,0.492188,0.087280,-0.687500,0.322021,-0.009773,-0.306396,...,-0.001689,0.001129,-0.003002,0.003481,0.002204,-0.004006,0.000914,-0.001327,-0.003738,-0.000344
4,doc_10962,1,60.0,terdakwa lahir bengkulu jenis kelamin laki lak...,0.494385,0.137939,-0.618164,0.332520,-0.020767,-0.322998,...,0.003581,-0.001232,-0.001316,-0.003664,-0.003823,0.000064,-0.000351,0.000571,-0.001696,0.001344


In [39]:
df.to_csv("fullReduced.csv", index=False)

In [40]:
df.head(5)

,id,lama hukuman (bulan),is_train,text,indobert_0,indobert_1,indobert_2,indobert_3,indobert_4,indobert_5,...,svd_1990,svd_1991,svd_1992,svd_1993,svd_1994,svd_1995,svd_1996,svd_1997,svd_1998,svd_1999
0,doc_13590,10.0,1,terdakwa terdakwa julianti lahinta alias jurni...,0.523438,0.229004,-0.600586,0.078857,0.000725,-0.063354,...,-0.011464,0.005271,-0.001099,-0.004057,-0.005500,-0.011825,0.007215,0.011113,0.000610,0.009561
1,doc_14914,60.0,1,terdakwa lahir yahadian jenis kelamin laki lak...,0.615234,0.380615,-0.715332,0.347900,-0.187988,-0.221191,...,-0.001062,0.007749,0.007159,0.003685,0.000923,-0.003474,0.012065,-0.001320,-0.023220,0.001949
2,doc_21900,18.0,1,acara pemeriksaan peradilan tingkat pertama me...,0.689453,-0.693848,-0.737305,0.776367,-0.001667,0.519531,...,0.011528,-0.009282,0.010786,-0.002834,-0.018224,0.000741,-0.005281,-0.000103,-0.015906,0.005294
3,doc_14859,72.0,1,terdakwa lahir manado jenis kelamin laki laki ...,0.492188,0.087280,-0.687500,0.322021,-0.009773,-0.306396,...,0.011220,-0.004495,-0.001441,-0.001880,0.000835,0.002843,0.004644,-0.004424,0.005769,0.008870
4,doc_10962,60.0,1,terdakwa lahir bengkulu jenis kelamin laki lak...,0.494385,0.137939,-0.618164,0.332520,-0.020767,-0.322998,...,0.008583,0.003431,0.006795,-0.003560,0.004260,0.006608,-0.000339,-0.002170,0.008578,0.015402


# MODELING

## Split

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23238 entries, 0 to 23237
Columns: 11028 entries, id to svd_9999
dtypes: float16(1024), float64(10001), int64(1), object(2)
memory usage: 1.8+ GB


In [57]:
# --- Split train & test ---
train_df = df[df["is_train"] == 1].drop(columns=["id", "text", "is_train"])
test_df  = df[df["is_train"] == 0].drop(columns=["id", "text", "is_train"])

X_train = train_df.drop(columns=["lama hukuman (bulan)"])
y_train = train_df["lama hukuman (bulan)"].astype(float)
X_test  = test_df.drop(columns=["lama hukuman (bulan)"])


## Train LGBM

In [58]:
# !pip install optuna
# !pip install -U scikit-learn


In [59]:
from sklearn.metrics import mean_squared_error
import numpy as np




In [ ]:
import lightgbm as lgb
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")  # silence warnings

def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "device": "gpu",              # 🔥 GPU
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", 4, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),   # L1
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0), # L2
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rmses = []

    for train_idx, valid_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model = lgb.LGBMRegressor(**params, n_estimators=500)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )

        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)

    return np.mean(rmses)

# --- Run Optuna with built-in progress bar ---
n_trials = 10
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best RMSE:", study.best_value)

# --- Train final model ---
best_params = study.best_params
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "device": "gpu",
    "gpu_platform_id": 0,
    "gpu_device_id": 0,
    "n_estimators": 500
})

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(X_train, y_train)

[I 2025-08-29 10:17:51,018] A new study created in memory with name: no-name-bbf274ed-e7a8-4b50-b4d7-b528aeb5137e


  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
import joblib

# Save
joblib.dump(final_model, "final_model.pkl")

# Load later
final_model = joblib.load("final_model.pkl")


## Test LGBM

In [ ]:
# --- Predict test set ---
preds = final_model.predict(X_test)

# --- Cap negatives to 0 ---
preds = np.where(preds < 0, 0, preds)   # or preds = np.clip(preds, 0, None)

# --- Save submission ---
submission = df[df["is_train"] == 0][["id"]].copy()
submission["lama hukuman (bulan)"] = preds
submission.to_csv("submission.csv", index=False)

submission.head()

## Train XGB